Raw table: AW_STG_BUS_DBO.IT_QUERY_HIST_ONE_MIN_AUDIT

Dedup table: AW_STG_BUS_DBO.IT_QUERY_HIST_ONE_MIN_AUDIT_DEDUP

Note: 
-Raw table data is being capture every minutes. Dedup table capture the dedup data. the dedup process runs 7:00 AM everyday. 

-The process excludes queries that encounter deadlock errors and small queries less than 5 minutes? 

-Dedup table following column combination will be unique APPLICATION_HANDLE,APPLICATION_NAME,SESSION_AUTH_ID,USER_EID,CLIENT_APPLNAME,STMT_TEXT

Column notes:
-DW_LOAD_TS is in UTC timezone
-Do not need Drop or Analyze
-Can not load DW_LOAD_TS


In [1]:
## This script reads log files, and parses out new tags
# coding: utf-8
import pandas as pd
import numpy as np
import sqlite3
import pypyodbc
import getpass
import re
import argparse
import os
import string
import csv
import random
from datetime import datetime

In [2]:
# Get EID and password so you can access AW
EIDU = input('Please enter your EID:')
EIDP = getpass.getpass('Please enter your Password: ')

Please enter your EID:knguy688
Please enter your Password: ········


In [3]:
#connect to ODBC driver
AWcxn = pypyodbc.connect(dsn = 'AWPROD', uid = EIDU, pwd = EIDP)

In [4]:
#aginityDW = pd.read_sql("SELECT CAST(DW_LOAD_TS AS TIMESTAMP) as DW_LOAD_TS FROM AW_STG_BUS_DBO.IT_QUERY_HIST_ONE_MIN_AUDIT_DEDUP limit 10", AWcxn)
#aginityDW

In [5]:
#Read in what we need directly
#aginity = pd.read_csv('bdeb416.csv', encoding = 'utf-8',low_memory=False, usecols=["Duration","Statement"])
#DW_LOAD_TS
#aginity = pd.read_sql("SELECT CAST(DW_LOAD_TS AS TIMESTAMP) as DW_LOAD_TS, APPLICATION_HANDLE,APPLICATION_NAME,SESSION_AUTH_ID,USER_EID,CLIENT_APPLNAME,ELAPSED_TIME_SEC,ACTIVITY_TYPE,TOTAL_CPU_TIME,ROWS_READ,ROWS_RETURNED,STMT_TEXT,QUERY_COST_ESTIMATE FROM AW_STG_BUS_DBO.IT_QUERY_HIST_ONE_MIN_AUDIT_DEDUP limit 10", AWcxn)

aginity = pd.read_sql("SELECT CAST(DW_LOAD_TS AS TIMESTAMP) as DW_LOAD_TS,APPLICATION_HANDLE,APPLICATION_NAME,SESSION_AUTH_ID,USER_EID,CLIENT_APPLNAME,ELAPSED_TIME_SEC,ACTIVITY_TYPE,TOTAL_CPU_TIME,ROWS_READ,ROWS_RETURNED,STMT_TEXT,QUERY_COST_ESTIMATE FROM AW_STG_BUS_DBO.IT_QUERY_HIST_ONE_MIN_AUDIT_DEDUP WHERE STMT_TEXT is not null", AWcxn)
aginity.head()



,dw_load_ts,application_handle,application_name,session_auth_id,user_eid,client_applname,elapsed_time_sec,activity_type,total_cpu_time,rows_read,rows_returned,stmt_text,query_cost_estimate
0,2019-09-04 07:14:30.669607,53,db2jcc_application,APPDSVCUSRAWPRD,n/a,None,16.0,READ_DML,0,0,0,select agent_id from TABLE( SNAP_GET_APPL_INFO...,1
1,2019-11-02 04:50:03.674407,90,db2jcc_application,APPDSVCUSRAWPRD,n/a,None,-62.0,READ_DML,0,0,0,"select SS.APPLICATION_HANDLE as Id, SS.SESSION...",23860
2,2019-11-10 18:02:46.787322,91,db2jcc_application,APPDSVCUSRAWPRD,n/a,None,-93.0,READ_DML,0,0,0,SELECT '(' CONCAT CAST( DBPARTITIONNUM as CHAR...,1
3,2019-09-21 08:54:03.554616,93,Aginity.HadoopWorkbe,EDUSRESS,ADMINIST,Aginity Workbench for Hadoop,5.0,WRITE_DML,0,0,0,insert into aw_cxa_dev.P5_bns_rdpt_FT_5PP_NAD_...,268
4,2019-09-28 13:32:03.289510,95,Aginity.HadoopWorkbe,EDUSRESS,ADMINIST,Aginity Workbench for Hadoop,763.0,WRITE_DML,630680600,0,0,insert into aw_cxa_dev.P7_nonpoint_FT_SCR_NAD_...,1743


In [6]:
aginity['ID'] = aginity.index
aginity.head()

,dw_load_ts,application_handle,application_name,session_auth_id,user_eid,client_applname,elapsed_time_sec,activity_type,total_cpu_time,rows_read,rows_returned,stmt_text,query_cost_estimate,ID
0,2019-09-04 07:14:30.669607,53,db2jcc_application,APPDSVCUSRAWPRD,n/a,None,16.0,READ_DML,0,0,0,select agent_id from TABLE( SNAP_GET_APPL_INFO...,1,0
1,2019-11-02 04:50:03.674407,90,db2jcc_application,APPDSVCUSRAWPRD,n/a,None,-62.0,READ_DML,0,0,0,"select SS.APPLICATION_HANDLE as Id, SS.SESSION...",23860,1
2,2019-11-10 18:02:46.787322,91,db2jcc_application,APPDSVCUSRAWPRD,n/a,None,-93.0,READ_DML,0,0,0,SELECT '(' CONCAT CAST( DBPARTITIONNUM as CHAR...,1,2
3,2019-09-21 08:54:03.554616,93,Aginity.HadoopWorkbe,EDUSRESS,ADMINIST,Aginity Workbench for Hadoop,5.0,WRITE_DML,0,0,0,insert into aw_cxa_dev.P5_bns_rdpt_FT_5PP_NAD_...,268,3
4,2019-09-28 13:32:03.289510,95,Aginity.HadoopWorkbe,EDUSRESS,ADMINIST,Aginity Workbench for Hadoop,763.0,WRITE_DML,630680600,0,0,insert into aw_cxa_dev.P7_nonpoint_FT_SCR_NAD_...,1743,4


In [7]:
# Define functions to be used 
def clean_query_string(sql_str):
    q = re.sub(r"\n|\t"," ", sql_str)
    return q.upper()

In [8]:
def tables_in_query(sql_str):

    # remove the /* */ comments
    q = re.sub(r"/\*[^*]*\*+(?:[^*/][^*]*\*+)*/", "", sql_str)

    # remove whole line -- and # comments
    lines = [line for line in q.splitlines() if not re.match("^\s*(--|#)", line)]

    # remove trailing -- and # comments
    q = " ".join([re.split("--|#", line)[0] for line in lines])

    # split on blanks, parens and semicolons
    tokens = re.split(r"[\s)(;]+", q)

    # scan the tokens. if we see a FROM or JOIN, we set the get_next
    # flag, and grab the next one (unless it's SELECT).

    result = set()
    get_next = False
    for tok in tokens:
        if get_next:
            if tok.lower() not in ["", "select", "if"]:
                result.add(tok.upper())
            get_next = False
        get_next = tok.lower() in ["from", "join", "table", "into", "exists", "view", "truncate"]

    return ','.join(result)

In [9]:
#Maybe also tag if the query includes “Create Hadoop table” or “insert into” or “Analyze Table”, or “select *” just with y/n flags.
def creates_in_query(sql_str):

    q = 'N'
    if re.search(r"create hadoop table", sql_str.lower()) :
        q = 'Y'

    return q

In [10]:
def inserts_in_query(sql_str):

    q = 'N'
    if re.search(r'insert into', sql_str.lower()) :
        q = 'Y'

    return q

In [11]:
def analyze_in_query(sql_str):

    q = 'N'
    if re.search(r'analyze table', sql_str.lower()) :
        q = 'Y'

    return q

def group_by_in_query(sql_str):

    q = 'N'
    if re.search(r'group by', sql_str.lower()) :
        q = 'Y'

    return q

In [12]:
def limit_in_query(sql_str):

    q = 'N'
    if re.search(r'fetch first', sql_str.lower()) or re.search(r'limit', sql_str.lower()) :
        q = 'Y'

    return q

In [13]:
def grant_in_query(sql_str):

    q = 'N'
    if re.search(r'grant', sql_str.lower()) :
        q = 'Y'

    return q

In [14]:
def tidy_split(df, column, sep='|', keep=False):
    """
    Split the values of a column and expand so the new DataFrame has one split
    value per row. Filters rows where the column is missing.

    Params
    ------
    df : pandas.DataFrame
        dataframe with the column to split and expand
    column : str
        the column to split and expand
    sep : str
        the string used to split the column's values
    keep : bool
        whether to retain the presplit value as it's own row

    Returns
    -------
    pandas.DataFrame
        Returns a dataframe with the same columns as `df`.
    """
    indexes = list()
    new_values = list()
    df = df.dropna(subset=[column])
    for i, presplit in enumerate(df[column].astype(str)):
        values = presplit.split(sep)
        if keep and len(values) > 1:
            indexes.append(i)
            new_values.append(presplit)
        for value in values:
            indexes.append(i)
            new_values.append(value)
    new_df = df.iloc[indexes, :].copy()
    new_values = re.sub(r" ","", new_values)
    new_df[column] = new_values
    return new_df

In [15]:
aginity['query_statement'] = aginity['stmt_text'].apply(clean_query_string)


In [16]:
#aginity['query_statement'] = aginity['stmt_text'].apply(clean_query_string)
aginity['tables'] = aginity['stmt_text'].apply(tables_in_query)
aginity['create_flag'] = aginity['stmt_text'].apply(creates_in_query)
aginity['insert_flag'] = aginity['stmt_text'].apply(inserts_in_query)
aginity['analyze_flag'] = aginity['stmt_text'].apply(analyze_in_query)
aginity['group_by_flag'] = aginity['stmt_text'].apply(group_by_in_query)
aginity['limit_flag'] = aginity['stmt_text'].apply(limit_in_query)
aginity['grant_flag'] = aginity['stmt_text'].apply(grant_in_query)
aginity.drop(columns=['stmt_text'], axis=1, inplace=True)

In [37]:
#fname, ext = os.path.splitext('bdeb416')
#aginity['username'] = fname.upper()

#aginity=tidy_split(aginity, 'tables', sep=',')

In [38]:
#write to 
#file_out = "{dir}/{name}_{out}{ext}".format(dir='out', name=fname, out='out', ext='.txt')
#aginity.to_csv('test', index=False, encoding='utf-8', sep ='\t')

In [17]:
pwd

'C:\\Users\\knguy688\\CX-Platforms-Support-COE\\AW projects\\Parsing SQL'

In [18]:
#aginity['tables'][0]

In [19]:
for col in aginity.columns:
    print (col)

dw_load_ts
application_handle
application_name
session_auth_id
user_eid
client_applname
elapsed_time_sec
activity_type
total_cpu_time
rows_read
rows_returned
query_cost_estimate
ID
query_statement
tables
create_flag
insert_flag
analyze_flag
group_by_flag
limit_flag
grant_flag


In [20]:
aginity.head()

,dw_load_ts,application_handle,application_name,session_auth_id,user_eid,client_applname,elapsed_time_sec,activity_type,total_cpu_time,rows_read,...,query_cost_estimate,ID,query_statement,tables,create_flag,insert_flag,analyze_flag,group_by_flag,limit_flag,grant_flag
0,2019-09-04 07:14:30.669607,53,db2jcc_application,APPDSVCUSRAWPRD,n/a,None,16.0,READ_DML,0,0,...,1,0,SELECT AGENT_ID FROM TABLE( SNAP_GET_APPL_INFO...,"SNAP_GET_APPL_INFO,TABLE",N,N,N,N,N,N
1,2019-11-02 04:50:03.674407,90,db2jcc_application,APPDSVCUSRAWPRD,n/a,None,-62.0,READ_DML,0,0,...,23860,1,"SELECT SS.APPLICATION_HANDLE AS ID, SS.SESSION...","SYSIBMADM.MON_CURRENT_SQL,SYSIBMADM.MON_LOCKWA...",N,N,N,N,N,N
2,2019-11-10 18:02:46.787322,91,db2jcc_application,APPDSVCUSRAWPRD,n/a,None,-93.0,READ_DML,0,0,...,1,2,SELECT '(' CONCAT CAST( DBPARTITIONNUM AS CHAR...,SYSIBMADM.DBCFG,N,N,N,N,N,N
3,2019-09-21 08:54:03.554616,93,Aginity.HadoopWorkbe,EDUSRESS,ADMINIST,Aginity Workbench for Hadoop,5.0,WRITE_DML,0,0,...,268,3,INSERT INTO AW_CXA_DEV.P5_BNS_RDPT_FT_5PP_NAD_...,AW_CXA_DEV.P5_BNS_RDPT_FT_5PP_NAD_LE730_MR_LAS...,N,Y,N,N,N,N
4,2019-09-28 13:32:03.289510,95,Aginity.HadoopWorkbe,EDUSRESS,ADMINIST,Aginity Workbench for Hadoop,763.0,WRITE_DML,630680600,0,...,1743,4,INSERT INTO AW_CXA_DEV.P7_NONPOINT_FT_SCR_NAD_...,AW_CXA_DEV.P7_NONPOINT_FT_SCR_NAD_GT730_MR_P7_...,N,Y,N,N,N,N


In [21]:
#parse delimited to rows
def explode(df, lst_cols, fill_value='', preserve_index=False):
    # make sure `lst_cols` is list-alike
    if (lst_cols is not None
        and len(lst_cols) > 0
        and not isinstance(lst_cols, (list, tuple, np.ndarray, pd.Series))):
        lst_cols = [lst_cols]
    # all columns except `lst_cols`
    idx_cols = df.columns.difference(lst_cols)
    # calculate lengths of lists
    lens = df[lst_cols[0]].str.len()
    # preserve original index values    
    idx = np.repeat(df.index.values, lens)
    # create "exploded" DF
    res = (pd.DataFrame({
                col:np.repeat(df[col].values, lens)
                for col in idx_cols},
                index=idx)
             .assign(**{col:np.concatenate(df.loc[lens>0, col].values)
                            for col in lst_cols}))
    # append those rows that have empty lists
    if (lens == 0).any():
        # at least one list in cells is empty
        res = (res.append(df.loc[lens==0, idx_cols], sort=False)
                  .fillna(fill_value))
    # revert the original index order
    res = res.sort_index()
    # reset index if requested
    if not preserve_index:        
        res = res.reset_index(drop=True)
    return res

In [22]:
df = aginity

In [23]:
#apply explode function
exaginity = explode(df.assign(tables=df.tables.str.split(',')), 'tables')
exaginity.head(n=10)

,ID,activity_type,analyze_flag,application_handle,application_name,client_applname,create_flag,dw_load_ts,elapsed_time_sec,grant_flag,...,insert_flag,limit_flag,query_cost_estimate,query_statement,rows_read,rows_returned,session_auth_id,total_cpu_time,user_eid,tables
0,0,READ_DML,N,53,db2jcc_application,None,N,2019-09-04 07:14:30.669607,16.0,N,...,N,N,1,SELECT AGENT_ID FROM TABLE( SNAP_GET_APPL_INFO...,0,0,APPDSVCUSRAWPRD,0,n/a,SNAP_GET_APPL_INFO
1,0,READ_DML,N,53,db2jcc_application,None,N,2019-09-04 07:14:30.669607,16.0,N,...,N,N,1,SELECT AGENT_ID FROM TABLE( SNAP_GET_APPL_INFO...,0,0,APPDSVCUSRAWPRD,0,n/a,TABLE
2,1,READ_DML,N,90,db2jcc_application,None,N,2019-11-02 04:50:03.674407,-62.0,N,...,N,N,23860,"SELECT SS.APPLICATION_HANDLE AS ID, SS.SESSION...",0,0,APPDSVCUSRAWPRD,0,n/a,SYSIBMADM.MON_CURRENT_SQL
3,1,READ_DML,N,90,db2jcc_application,None,N,2019-11-02 04:50:03.674407,-62.0,N,...,N,N,23860,"SELECT SS.APPLICATION_HANDLE AS ID, SS.SESSION...",0,0,APPDSVCUSRAWPRD,0,n/a,SYSIBMADM.MON_LOCKWAITS
4,1,READ_DML,N,90,db2jcc_application,None,N,2019-11-02 04:50:03.674407,-62.0,N,...,N,N,23860,"SELECT SS.APPLICATION_HANDLE AS ID, SS.SESSION...",0,0,APPDSVCUSRAWPRD,0,n/a,TABLE


In [24]:
exaginity['tables'][2]

'SYSIBMADM.MON_CURRENT_SQL'

In [26]:
exaginity['clean_tables'] = aginity['tables'].apply(clean_query_string)

In [29]:
exaginity.head(n=10)

,ID,activity_type,analyze_flag,application_handle,application_name,client_applname,create_flag,dw_load_ts,elapsed_time_sec,grant_flag,...,limit_flag,query_cost_estimate,query_statement,rows_read,rows_returned,session_auth_id,total_cpu_time,user_eid,tables,clean_tables
0,0,READ_DML,N,53,db2jcc_application,None,N,2019-09-04 07:14:30.669607,16.0,N,...,N,1,SELECT AGENT_ID FROM TABLE( SNAP_GET_APPL_INFO...,0,0,APPDSVCUSRAWPRD,0,n/a,SNAP_GET_APPL_INFO,"SNAP_GET_APPL_INFO,TABLE"
1,0,READ_DML,N,53,db2jcc_application,None,N,2019-09-04 07:14:30.669607,16.0,N,...,N,1,SELECT AGENT_ID FROM TABLE( SNAP_GET_APPL_INFO...,0,0,APPDSVCUSRAWPRD,0,n/a,TABLE,"SYSIBMADM.MON_CURRENT_SQL,SYSIBMADM.MON_LOCKWA..."
2,1,READ_DML,N,90,db2jcc_application,None,N,2019-11-02 04:50:03.674407,-62.0,N,...,N,23860,"SELECT SS.APPLICATION_HANDLE AS ID, SS.SESSION...",0,0,APPDSVCUSRAWPRD,0,n/a,SYSIBMADM.MON_CURRENT_SQL,SYSIBMADM.DBCFG
3,1,READ_DML,N,90,db2jcc_application,None,N,2019-11-02 04:50:03.674407,-62.0,N,...,N,23860,"SELECT SS.APPLICATION_HANDLE AS ID, SS.SESSION...",0,0,APPDSVCUSRAWPRD,0,n/a,SYSIBMADM.MON_LOCKWAITS,AW_CXA_DEV.P5_BNS_RDPT_FT_5PP_NAD_LE730_MR_LAS...
4,1,READ_DML,N,90,db2jcc_application,None,N,2019-11-02 04:50:03.674407,-62.0,N,...,N,23860,"SELECT SS.APPLICATION_HANDLE AS ID, SS.SESSION...",0,0,APPDSVCUSRAWPRD,0,n/a,TABLE,AW_CXA_DEV.P7_NONPOINT_FT_SCR_NAD_GT730_MR_P7_...
5,1,READ_DML,N,90,db2jcc_application,None,N,2019-11-02 04:50:03.674407,-62.0,N,...,N,23860,"SELECT SS.APPLICATION_HANDLE AS ID, SS.SESSION...",0,0,APPDSVCUSRAWPRD,0,n/a,MON_GET_CONNECTION,AW_CXA_DEV.P7_NONPOINT_FT_MTP_NAD_GT730_MR_P7_...
6,2,READ_DML,N,91,db2jcc_application,None,N,2019-11-10 18:02:46.787322,-93.0,N,...,N,1,SELECT '(' CONCAT CAST( DBPARTITIONNUM AS CHAR...,0,0,APPDSVCUSRAWPRD,0,n/a,SYSIBMADM.DBCFG,"SYSIBMADM.MON_CURRENT_SQL,SYSIBMADM.MON_LOCKWA..."
7,3,WRITE_DML,N,93,Aginity.HadoopWorkbe,Aginity Workbench for Hadoop,N,2019-09-21 08:54:03.554616,5.0,N,...,N,268,INSERT INTO AW_CXA_DEV.P5_BNS_RDPT_FT_5PP_NAD_...,0,0,EDUSRESS,0,ADMINIST,AW_CXA_DEV.P5_BNS_RDPT_FT_5PP_NAD_LE730_MR_LAS...,AW_CXA_DEV.P12_ST_RVNU_FT_MTP_NAD_GT730_MR_HTL...
8,3,WRITE_DML,N,93,Aginity.HadoopWorkbe,Aginity Workbench for Hadoop,N,2019-09-21 08:54:03.554616,5.0,N,...,N,268,INSERT INTO AW_CXA_DEV.P5_BNS_RDPT_FT_5PP_NAD_...,0,0,EDUSRESS,0,ADMINIST,AW_CXA_DEV.P5_BNS_RDPT_FT_5PP_NAD_LE730_MR_LAS...,AW_CXA_DEV.P12_ST_RVNU_FT_MTP_NAD_GT730_MR_HTL...
9,4,WRITE_DML,N,95,Aginity.HadoopWorkbe,Aginity Workbench for Hadoop,N,2019-09-28 13:32:03.289510,763.0,N,...,N,1743,INSERT INTO AW_CXA_DEV.P7_NONPOINT_FT_SCR_NAD_...,0,0,EDUSRESS,630680600,ADMINIST,AW_CXA_DEV.P7_NONPOINT_FT_SCR_NAD_GT730_MR_P7_...,AW_CXA_DEV.P12_ST_RVNU_FT_MTP_NAD_GT730_MR_HTL...


In [25]:
#write to 
#file_out = "{dir}/{name}_{out}{ext}".format(dir='out', name=fname, out='out', ext='.txt')
exaginity.to_csv('test', index=False, encoding='utf-8', sep ='\t')

Drop table if exists knguy688.userquery;

CREATE HADOOP TABLE KNGUY688.USERQUERY(
	ID SMALLINT,
	ACTIVITY_TYPE VARCHAR(32),
	ANALYZE_FLAG VARCHAR(25),
	APPLICATION_HANDLE BIGINT,
	APPLICATION_NAME VARCHAR(255),
	CLIENT_APPLNAME VARCHAR(255),
	CREATE_FLAG VARCHAR(25),
	DW_LOAD_TS	TIMESTAMP,
	ELAPSED_TIME_SEC INTEGER,
	GRANT_FLAG VARCHAR(25),
	GROUP_BY_FLAG VARCHAR(25),
	INSERT_FLAG VARCHAR(25),	
	LIMIT_FLAG VARCHAR(25),
	QUERY_COST_ESTIMATE BIGINT,
	QUERY_STATEMENT VARCHAR(3200),
	ROWS_READ BIGINT,
	ROWS_RETURNED BIGINT,	
	SESSION_AUTH_ID VARCHAR(255),
	TOTAL_CPU_TIME BIGINT,
	USER_EID VARCHAR(255),	
	TABLES VARCHAR(3200)
);

In [22]:
#Truncate
exaginity.replace(to_replace=[None], value='N/A', inplace=True)
exaginity = exaginity.drop(['query_statement'], axis=1)
cur=AWcxn.cursor()
truncateSQl = "TRUNCATE TABLE KNGUY688.USERQUERY;"
cur.execute(truncateSQl)

#Insert Procedure
list_of_dictionaries=exaginity.T.to_dict().values()

for i in list_of_dictionaries:
    insert_sql="INSERT INTO KNGUY688.USERQUERY (ID, ACTIVITY_TYPE, ANALYZE_FLAG, APPLICATION_HANDLE, APPLICATION_NAME, CLIENT_APPLNAME, CREATE_FLAG, DW_LOAD_TS, ELAPSED_TIME_SEC, GRANT_FLAG, GROUP_BY_FLAG, INSERT_FLAG, LIMIT_FLAG, QUERY_COST_ESTIMATE, ROWS_READ, ROWS_RETURNED, SESSION_AUTH_ID, TOTAL_CPU_TIME, USER_EID, TABLES) VALUES %s"%str(tuple(i.values()))
    cur.execute(insert_sql)
print(len(list_of_dictionaries), "Records inserted") 
 
AWcxn.close()

DatabaseError: ('40003', '[40003] [IBM][CLI Driver] SQL30081N  A communication error has been detected. Communication protocol being used: "TCP/IP".  Communication API being used: "SOCKETS".  Location where the error was detected: "172.20.68.246".  Communication function detecting the error: "recv".  Protocol specific error code(s): "10060", "*", "*".  SQLSTATE=08001\r\n')

In [ ]:
#write to 
#file_out = "{dir}/{name}_{out}{ext}".format(dir='out', name=fname, out='out', ext='.txt')
exaginity.to_csv('test', index=False, encoding='utf-8', sep ='\t')

In [67]:
maxColumnLenghts = []
for col in range(len(exaginity.columns)):
    maxColumnLenghts.append(max(exaginity.iloc[:,col].astype(str).apply(len)))
print('Max Column Lengths ', maxColumnLenghts)

Max Column Lengths  [1, 9, 1, 2, 20, 28, 1, 3, 1, 1, 1, 1, 6, 1958, 1, 1, 15, 5, 8, 70]


In [100]:
for col in exaginity.columns:
    print (col)

ID
activity_type
analyze_flag
application_handle
application_name
client_applname
create_flag
dw_load_ts
elapsed_time_sec
grant_flag
group_by_flag
insert_flag
limit_flag
query_cost_estimate
query_statement
rows_read
rows_returned
session_auth_id
total_cpu_time
user_eid
tables


In [112]:
#use this to this in aginity - remove "" and / using replace
insert_sql

"INSERT INTO KNGUY688.USERQUERY (ID, ACTIVITY_TYPE, ANALYZE_FLAG, APPLICATION_HANDLE, APPLICATION_NAME, CLIENT_APPLNAME, CREATE_FLAG, DW_LOAD_TS, ELAPSED_TIME_SEC, GRANT_FLAG, GROUP_BY_FLAG, INSERT_FLAG, LIMIT_FLAG, QUERY_COST_ESTIMATE, ROWS_READ, ROWS_RETURNED, SESSION_AUTH_ID, TOTAL_CPU_TIME, USER_EID, TABLES) VALUES (9, 'CALL', 'N', 99, 'Aginity.HadoopWorkbe', 'Aginity Workbench for Hadoop', 'N', Timestamp('2019-09-27 07:53:44.084227'), 143, 'N', 'N', 'N', 'N', -1, 0, 0, 'EDUSRESS', 1557, 'ADMINIST', 'AW_CXA_DEV.FT_SCR_ROW_GT730_MR_P2_ADR')"